# Neo4j Graph Analytics on Snowflake

**Prerequisite:** Neo4j Graph Analytics installed from Snowflake Marketplace  
**Data:** Process mining node/relationship tables from Notebook 02  
**Features:** Community detection (WCC, Louvain), PageRank, similarity — all via SQL

In [ ]:
import snowflake.connector
import os
import pandas as pd
import plotly.express as px

conn = snowflake.connector.connect(connection_name=os.getenv('SNOWFLAKE_CONNECTION_NAME') or 'AU_DEMO50')
cur = conn.cursor()
cur.execute('USE DATABASE CDSB_DEMO')
cur.execute('USE SCHEMA RAW')
cur.execute('USE WAREHOUSE COMPUTE_WH')
print('Connected')

## 1. Setup — Grant Neo4j Access to Our Data
Neo4j runs as a Native App inside Snowflake — your data never leaves

In [ ]:
setup_sql = """
-- Grant Neo4j access to read our tables and write results
-- (Run once — already done if you've set this up before)

USE ROLE ACCOUNTADMIN;

-- Create database role for Neo4j access
CREATE DATABASE ROLE IF NOT EXISTS CDSB_DEMO.NEO4J_ROLE;
GRANT USAGE ON DATABASE CDSB_DEMO TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT USAGE ON SCHEMA CDSB_DEMO.RAW TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT SELECT ON ALL TABLES IN SCHEMA CDSB_DEMO.RAW TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT SELECT ON FUTURE TABLES IN SCHEMA CDSB_DEMO.RAW TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT CREATE TABLE ON SCHEMA CDSB_DEMO.RAW TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT DATABASE ROLE CDSB_DEMO.NEO4J_ROLE TO APPLICATION Neo4j_Graph_Analytics;
"""

for stmt in [s.strip() for s in setup_sql.split(';') if s.strip() and not s.strip().startswith('--')]:
    try:
        cur.execute(stmt)
        print(f'✓ {stmt[:60]}...')
    except Exception as e:
        print(f'⚠ {stmt[:60]}... → {e}')

In [ ]:
df_nodes = pd.read_sql("SELECT NODE_TYPE, COUNT(*) as n FROM PROCESS_NODES GROUP BY NODE_TYPE", conn)
df_rels = pd.read_sql("SELECT relationshipType, COUNT(*) as n FROM PROCESS_RELATIONSHIPS GROUP BY relationshipType", conn)
print("Graph Structure:")
print(f"  Nodes: {df_nodes.to_dict('records')}")
print(f"  Relationships: {df_rels.to_dict('records')}")
print(f"\nTotal: {df_nodes['N'].sum()} nodes, {df_rels['N'].sum()} relationships")

## 2. Community Detection — Weakly Connected Components
Find disconnected groups of cases and handlers

In [ ]:
cur.execute("""
    CALL Neo4j_Graph_Analytics.graph.wcc('CPU_X64_XS', {
        'project': {
            'nodeTables': ['CDSB_DEMO.RAW.PROCESS_NODES'],
            'relationshipTables': {
                'CDSB_DEMO.RAW.PROCESS_RELATIONSHIPS': {
                    'sourceTable': 'CDSB_DEMO.RAW.PROCESS_NODES',
                    'targetTable': 'CDSB_DEMO.RAW.PROCESS_NODES',
                    'orientation': 'UNDIRECTED'
                }
            }
        },
        'compute': { 'consecutiveIds': true },
        'write': [{
            'nodeLabel': 'PROCESS_NODES',
            'outputTable': 'CDSB_DEMO.RAW.NODES_WCC_RESULTS'
        }]
    })
""")
print("WCC complete!")

df_wcc = pd.read_sql("""
    SELECT VALUE as COMPONENT, COUNT(*) as NODES
    FROM CDSB_DEMO.RAW.NODES_WCC_RESULTS
    GROUP BY VALUE
    ORDER BY NODES DESC
    LIMIT 10
""", conn)
print("\nTop 10 Connected Components:")
print(df_wcc.to_string(index=False))

## 3. PageRank — Find the Most Central Handlers
Who are the most connected / influential actors in the service delivery network?

In [ ]:
cur.execute("""
    CALL Neo4j_Graph_Analytics.graph.pagerank('CPU_X64_XS', {
        'project': {
            'nodeTables': ['CDSB_DEMO.RAW.PROCESS_NODES'],
            'relationshipTables': {
                'CDSB_DEMO.RAW.PROCESS_RELATIONSHIPS': {
                    'sourceTable': 'CDSB_DEMO.RAW.PROCESS_NODES',
                    'targetTable': 'CDSB_DEMO.RAW.PROCESS_NODES',
                    'orientation': 'NATURAL'
                }
            }
        },
        'compute': { 'maxIterations': 20, 'dampingFactor': 0.85 },
        'write': [{
            'nodeLabel': 'PROCESS_NODES',
            'outputTable': 'CDSB_DEMO.RAW.NODES_PAGERANK_RESULTS'
        }]
    })
""")
print("PageRank complete!")

df_pr = pd.read_sql("""
    SELECT pr.NODE, ROUND(pr.VALUE, 6) as PAGERANK, n.NODE_ID, n.NODE_TYPE
    FROM CDSB_DEMO.RAW.NODES_PAGERANK_RESULTS pr
    JOIN CDSB_DEMO.RAW.PROCESS_NODES n ON pr.NODE = n.nodeId
    WHERE n.NODE_TYPE IN ('Handler', 'Region')
    ORDER BY pr.VALUE DESC
    LIMIT 15
""", conn)
print("\nMost Central Actors (PageRank):")
print(df_pr.to_string(index=False))

fig = px.bar(df_pr, x='NODE_ID', y='PAGERANK', color='NODE_TYPE',
             title='PageRank: Most Central Actors in Service Delivery Network')
fig.update_layout(template='plotly_white')
fig.show()

## 4. Louvain Community Detection
Find tightly-connected communities of handlers and cases

In [ ]:
cur.execute("""
    CALL Neo4j_Graph_Analytics.graph.louvain('CPU_X64_XS', {
        'project': {
            'nodeTables': ['CDSB_DEMO.RAW.PROCESS_NODES'],
            'relationshipTables': {
                'CDSB_DEMO.RAW.PROCESS_RELATIONSHIPS': {
                    'sourceTable': 'CDSB_DEMO.RAW.PROCESS_NODES',
                    'targetTable': 'CDSB_DEMO.RAW.PROCESS_NODES',
                    'orientation': 'UNDIRECTED'
                }
            }
        },
        'compute': {},
        'write': [{
            'nodeLabel': 'PROCESS_NODES',
            'outputTable': 'CDSB_DEMO.RAW.NODES_LOUVAIN_RESULTS'
        }]
    })
""")
print("Louvain community detection complete!")

df_communities = pd.read_sql("""
    SELECT l.VALUE as COMMUNITY, n.NODE_TYPE, COUNT(*) as MEMBERS
    FROM CDSB_DEMO.RAW.NODES_LOUVAIN_RESULTS l
    JOIN CDSB_DEMO.RAW.PROCESS_NODES n ON l.NODE = n.nodeId
    GROUP BY l.VALUE, n.NODE_TYPE
    ORDER BY MEMBERS DESC
    LIMIT 20
""", conn)
print("\nCommunities Detected:")
print(df_communities.to_string(index=False))

fig = px.bar(df_communities, x='COMMUNITY', y='MEMBERS', color='NODE_TYPE',
             title='Louvain Communities: Handler-Case Clusters')
fig.update_layout(template='plotly_white', xaxis_title='Community ID')
fig.show()

## 5. Combine Graph Features with Process Mining Results
Join community/centrality data back to original cases for deeper analysis

In [ ]:
df_enriched = pd.read_sql("""
    WITH case_summary AS (
        SELECT CASE_ID,
               LISTAGG(ACTIVITY, ' → ') WITHIN GROUP (ORDER BY EVENT_TIME) as PATH,
               DATEDIFF('hour', MIN(EVENT_TIME), MAX(EVENT_TIME)) as DURATION_HOURS,
               MAX(CHANNEL) as CHANNEL,
               MAX(REQUEST_TYPE) as REQUEST_TYPE,
               MAX(ACTOR) as HANDLER
        FROM SERVICE_REQUEST_EVENTS
        WHERE ACTOR NOT IN ('System', 'Triage_Team', 'Escalation_Manager')
        GROUP BY CASE_ID
    ),
    handler_rank AS (
        SELECT n.NODE_ID as HANDLER, ROUND(pr.VALUE, 6) as PAGERANK
        FROM NODES_PAGERANK_RESULTS pr
        JOIN PROCESS_NODES n ON pr.NODE = n.nodeId
        WHERE n.NODE_TYPE = 'Handler'
    )
    SELECT cs.*, hr.PAGERANK as HANDLER_CENTRALITY
    FROM case_summary cs
    LEFT JOIN handler_rank hr ON cs.HANDLER = hr.HANDLER
    ORDER BY DURATION_HOURS DESC
    LIMIT 20
""", conn)

print("Enriched Case Data (graph features + process mining):")
df_enriched

## Key Takeaways

| Algorithm | Use Case | What It Found |
|-----------|----------|--------------|
| WCC | Find disconnected case groups | Isolated case clusters that may indicate process silos |
| PageRank | Identify key handlers | Most central actors in service delivery |
| Louvain | Community detection | Handler-case communities revealing team dynamics |

**Neo4j Graph Analytics runs inside Snowflake as a Native App**  
- No data movement — your data stays in Snowflake  
- SQL interface — no Cypher needed  
- Results write back to Snowflake tables for joining, dashboarding, ML  
- Perfect for fraud detection, entity resolution, network analysis